<a href="https://colab.research.google.com/github/pinomartinezruben/poker-bot/blob/main/ruben_pokerbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# -----------------------
# Card helpers
# -----------------------

rank_names = {
    2: "2", 3: "3", 4: "4", 5: "5", 6: "6",
    7: "7", 8: "8", 9: "9", 10: "10",
    11: "Jack", 12: "Queen", 13: "King", 14: "Ace"
}

suit_names = {
    0: "Hearts",
    1: "Diamonds",
    2: "Clubs",
    3: "Spades"
}

# -----------------------
# Describe Hand -- Functiom that describes the hand the user has. Used at the end of this program, to assist print statements describing hand.
# -----------------------

def describe_hand(hand):
    r1, s1, r2, s2 = hand

    card1 = f"{rank_names[r1]} of {suit_names[s1]}"
    card2 = f"{rank_names[r2]} of {suit_names[s2]}"

    description = []

    if r1 == r2:
        description.append("Pair")

    if s1 == s2:
        description.append("Suited")

    if max(r1, r2) >= 11:
        description.append("High Card")

    if not description:
        description.append("Low Hand")

    return card1, card2, ", ".join(description)


# -----------------------
# Generate synthetic data
# -----------------------

def generate_hand():
    rank1 = np.random.randint(2, 15)
    rank2 = np.random.randint(2, 15)
    suit1 = np.random.randint(0, 4)
    suit2 = np.random.randint(0, 4)

    return [rank1, suit1, rank2, suit2]


# -----------------------
# Calculte hand strength -- Algrorithm that calculates the hand strength using multiple variables, including card/suit pairs and a grade for how high the card values are.
# -----------------------


def hand_strength(hand):
    r1, s1, r2, s2 = hand

    strength = 0

    # Pair bonus
    if r1 == r2:
        strength += 5

    # High card value grade
    strength += (r1 + r2) / 28

    # Suit bonus
    if s1 == s2:
        strength += 1

    return strength


# -----------------------
# Calculate optimal bet -- Calculate the optimal bet using the calculated hand strength, with (possibly) a random value to simulate other random game conditions (opponent bluffs, dealer cards, etc)
# -----------------------


def optimal_bet(strength):
    return strength * 20 # + np.random.normal(0, 2)


# -----------------------
# Create Dataset
# -----------------------

X = []
y = []

for _ in range(10000):
    hand = generate_hand()
    strength = hand_strength(hand)
    bet = optimal_bet(strength)

    X.append(hand)
    y.append(bet)

X = np.array(X, dtype=float)
y = np.array(y, dtype=float)

# Normalize
X_max = np.max(X, axis=0)
X = X / X_max


# -----------------------
# Build linear regression model
# -----------------------

model = keras.Sequential([
    layers.Dense(16, input_shape=(4,)),
    layers.Dense(32, activation='silu'),
    layers.Dense(64, activation='silu'),
    layers.Dense(32, activation='silu'),
    layers.Dense(16, activation='silu'),
    layers.Dense(1)
])

model.compile(
    optimizer="adamW",
    loss="mse",
    metrics=["mae"]
)

model.fit(X, y, epochs=100, batch_size=32, validation_split=0.2)


# -----------------------
# Test prediction
# -----------------------

def test_prediction(test_hand):

  card1, card2, desc = describe_hand(test_hand)

  strength = hand_strength(test_hand)

  test_array = np.array([test_hand], dtype=float) / X_max
  prediction = model.predict(test_array)[0][0]


  print("\n==============================")
  print("POKER HAND ANALYSIS")
  print("==============================")

  print(f"Card 1: {card1}")
  print(f"Card 2: {card2}")
  print(f"Hand Type: {desc}")

  print("\nEstimated Hand Strength:", round(strength, 2))
  print("Recommended Bet Amount: $", round(prediction, 2))

  print("==============================\n")


test_hand_1 = [14, 1, 14, 3]  # Pocket Aces example
test_hand_2 = [5, 1, 6, 1]   # Example 2

test_hands = []
for i in range(2, 15):
  test_hands.append([i, 0, i, 2])
  test_prediction(test_hands[i-2])

for i in range(3, 9):
  test_hands.append([i-1, 0, i, 2])
  test_prediction(test_hands[i-2])

# test_prediction(test_hand_1)
# test_prediction(test_hand_2)



Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1233.2140 - mae: 21.3571 - val_loss: 701.4172 - val_mae: 16.3422
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 784.8343 - mae: 17.2863 - val_loss: 695.8366 - val_mae: 16.3913
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 797.3483 - mae: 17.4919 - val_loss: 703.9940 - val_mae: 17.4665
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 806.5837 - mae: 17.9016 - val_loss: 692.3087 - val_mae: 16.5794
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 839.7028 - mae: 18.1451 - val_loss: 689.5402 - val_mae: 16.0392
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 778.6051 - mae: 17.3724 - val_loss: 690.4878 - val_mae: 17.0190
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 790.2281 - mae: 17.6566 - val_loss: 681.7852 - val_mae: 16.6166
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 750.9081 - mae: 16.9994 - val_loss: 670.0462 - val_mae: 15.3978